# 📓 Exercise 02 — BM25: Best Match 25

**Series:** RAG Foundations | **Difficulty:** ⭐⭐ Beginner-Intermediate  
**Prerequisites:** Exercise 01 (TF-IDF)  
**Time to complete:** ~35 minutes

---

## 🎯 Learning Objectives
1. Understand the two problems with TF-IDF that BM25 solves
2. Explain **term saturation** and why it matters
3. Explain **document length normalisation** and why it matters
4. Compute BM25 scores manually and with the `rank-bm25` library
5. Compare TF-IDF vs BM25 on edge cases

---

## 📖 Concept: Why BM25 Over TF-IDF?

TF-IDF has two weaknesses that BM25 was specifically designed to fix:

### Problem 1: Term Saturation
> TF-IDF says: if `"moon"` appears **10×** in a document, it's **10× more relevant** than a doc where it appears **1×**.

This is unrealistic. A document where "moon" appears 10 times isn't **10× more relevant** than one where it appears once — the extra repetitions add diminishing information.

**BM25's fix:** A saturation function that "caps" the effect of repeated terms.

### Problem 2: Document Length Bias
> TF-IDF's normalisation (dividing by doc length) doesn't fully account for varying lengths.
> A 1000-word document can still score higher just because it repeats query terms more often.

**BM25's fix:** A more nuanced length normalisation that uses the *average* document length.

---

## 📐 BM25 Formula

```
BM25(w, d) = IDF(w) × [ f(w,d) × (k+1) ] / [ f(w,d) + k × (1 - b + b × |d|/avgdl) ]

Where:
  f(w, d)  = raw count of word w in document d
  |d|      = document length (word count)
  avgdl    = average document length across the whole corpus
  k        = term saturation parameter (typically 1.5 or 2.0)
  b        = length normalisation strength (0=no normalisation, 1=full, typically 0.75)
```

**BM25 IDF (slightly different from TF-IDF):**
```
IDF(w) = log( (N - df(w) + 0.5) / (df(w) + 0.5) + 1 )
```

---

In [ ]:
!pip install rank-bm25 numpy pandas matplotlib scikit-learn --quiet

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Libraries loaded!")

---
## 🔬 Part 1: Term Saturation — BM25's Key Innovation

### Visualising the Saturation Effect

In [ ]:
# ── Visualise term saturation: TF-IDF vs BM25 ──

k = 1.5   # BM25 k parameter
b = 0.75  # BM25 b parameter
avgdl = 10  # assume average doc length = 10 words
dl = 10     # this doc has exactly 10 words (equal to average → no length penalty)
idf = 2.0   # fixed IDF value for illustration

term_counts = np.arange(0, 21)  # word appears 0 to 20 times

# TF-IDF: score grows linearly with count
tfidf_scores = (term_counts / dl) * idf

# BM25: score saturates
bm25_scores = idf * (term_counts * (k + 1)) / (term_counts + k * (1 - b + b * dl / avgdl))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw score comparison
axes[0].plot(term_counts, tfidf_scores, 'o-', color='#E74C3C', label='TF-IDF (linear growth)', linewidth=2)
axes[0].plot(term_counts, bm25_scores,  's-', color='#27AE60', label='BM25 (saturates)', linewidth=2)
axes[0].axhline(idf * (k+1) / 1, color='#27AE60', linestyle='--', alpha=0.4, label='BM25 ceiling')
axes[0].set_xlabel('Number of times word appears in document', fontsize=10)
axes[0].set_ylabel('Contribution to score', fontsize=10)
axes[0].set_title('Term Saturation: TF-IDF vs BM25', fontsize=11)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Marginal gain (how much EXTRA does each additional occurrence add?)
tfidf_marginal = np.diff(tfidf_scores)
bm25_marginal  = np.diff(bm25_scores)
axes[1].plot(term_counts[1:], tfidf_marginal, 'o-', color='#E74C3C', label='TF-IDF (constant gain)', linewidth=2)
axes[1].plot(term_counts[1:], bm25_marginal,  's-', color='#27AE60', label='BM25 (diminishing returns)', linewidth=2)
axes[1].set_xlabel('Term count', fontsize=10)
axes[1].set_ylabel('Marginal gain from one more occurrence', fontsize=10)
axes[1].set_title('Marginal Value of Additional Term Occurrences', fontsize=11)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('BM25 Term Saturation vs TF-IDF Linear Growth', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('bm25_saturation.png', dpi=120, bbox_inches='tight')
plt.show()
print("💾 Saved: bm25_saturation.png")
print("\n📝 Key insight: In BM25, the 10th occurrence adds much less than the 1st.")
print("   In TF-IDF, every occurrence adds exactly the same amount (linear).")

### ✏️ Exercise 1.1 — Effect of the k parameter

The `k` parameter controls *how fast* saturation happens:
- **k = 0**: complete saturation (only the first occurrence matters!)
- **k = large (e.g., 100)**: almost no saturation (behaves like TF-IDF)
- **k = 1.5**: typical BM25 default — good balance

Run the cell below to explore this:

In [ ]:
# ✏️ Exercise 1.1 — k parameter exploration

k_values = [0.5, 1.2, 1.5, 2.0, 10.0]  # Try different k values
term_counts = np.arange(0, 16)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#E74C3C', '#E67E22', '#27AE60', '#4A90D9', '#8E44AD']

for k_val, color in zip(k_values, colors):
    bm25_scores = 2.0 * (term_counts * (k_val + 1)) / (term_counts + k_val)
    ax.plot(term_counts, bm25_scores, 'o-', color=color, linewidth=2, 
            label=f'k={k_val}', alpha=0.8)

ax.set_xlabel('Term frequency (raw count)', fontsize=11)
ax.set_ylabel('BM25 TF component', fontsize=11)
ax.set_title('Effect of k on Term Saturation\n(lower k = faster saturation)', fontsize=11)
ax.legend(title='k parameter')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Observations:")
print("  k=0.5  → saturates very quickly (even 3 occurrences ≈ same as 15)")
print("  k=1.5  → standard default — good balance (recommended)")
print("  k=10.0 → barely any saturation — behaves almost like TF × IDF")

---
## 🔬 Part 2: Document Length Normalisation

### The Problem BM25 Solves

Consider two documents:
- **Short Doc**: `"moon mission"` (2 words)
- **Long Doc**: `"the apollo moon mission was a great moon mission that went to the moon"` (15 words)

The long doc has "moon" and "mission" more times, so TF-IDF might rank it higher — but is it really more relevant? BM25 **penalises** the long doc proportionally.

In [ ]:
# ── DEMO: Length normalisation effect ──

b_values = [0.0, 0.25, 0.5, 0.75, 1.0]
k = 1.5
idf = 2.0
f = 3      # word appears 3 times
avgdl = 10 # average doc length is 10 words

# Docs of different lengths all containing the word 3 times
doc_lengths = np.arange(1, 41)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#E74C3C', '#E67E22', '#27AE60', '#4A90D9', '#8E44AD']

for b_val, color in zip(b_values, colors):
    scores = idf * f * (k+1) / (f + k * (1 - b_val + b_val * doc_lengths / avgdl))
    axes[0].plot(doc_lengths, scores, linewidth=2, color=color, label=f'b={b_val}', alpha=0.85)

axes[0].axvline(avgdl, color='gray', linestyle='--', alpha=0.6, label=f'avgdl={avgdl}')
axes[0].set_xlabel('Document Length (words)', fontsize=10)
axes[0].set_ylabel('BM25 Score', fontsize=10)
axes[0].set_title('Effect of b on Length Normalisation\n(word appears 3 times in each doc)', fontsize=10)
axes[0].legend(title='b parameter')
axes[0].grid(alpha=0.3)

# Side-by-side example
scenarios = [
    ("Short doc (5w)",   5,  3),
    ("Average doc (10w)",10,  3),
    ("Long doc (30w)",   30,  3),
    ("Long doc (30w, 9 mentions)", 30, 9),
]
b = 0.75
scores_noB = [idf * f_val*(k+1)/(f_val+k) for _, _, f_val in scenarios]
scores_withB = [idf * f_val*(k+1)/(f_val + k*(1-b + b*dl/avgdl)) 
                for _, dl, f_val in scenarios]

x = np.arange(len(scenarios))
axes[1].bar(x - 0.2, scores_noB,   0.4, label='b=0 (no length norm)', color='#E74C3C', alpha=0.85)
axes[1].bar(x + 0.2, scores_withB, 0.4, label='b=0.75 (recommended)', color='#27AE60', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels([s[0] for s in scenarios], rotation=20, ha='right', fontsize=8)
axes[1].set_ylabel('BM25 Score', fontsize=10)
axes[1].set_title('With vs Without Length Normalisation', fontsize=10)
axes[1].legend(fontsize=9)

plt.suptitle('BM25 Document Length Normalisation (b parameter)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('bm25_length_norm.png', dpi=120, bbox_inches='tight')
plt.show()
print("💾 Saved: bm25_length_norm.png")

---
## 🔬 Part 3: Implementing BM25 from Scratch

In [ ]:
# ── BM25 Class — fully documented implementation ──

class BM25:
    """
    BM25 (Best Match 25) retrieval model.
    
    An improved keyword search ranking function over TF-IDF.
    Addresses term saturation and document length bias.
    
    Parameters:
        k : float — term saturation parameter (default 1.5)
                    Lower = faster saturation, higher = more linear
        b : float — length normalisation parameter (default 0.75)
                    0 = no normalisation, 1 = full normalisation
    """
    
    def __init__(self, documents, k=1.5, b=0.75):
        self.k = k
        self.b = b
        self.N = len(documents)
        
        # Tokenise all documents
        self.tokenised_docs = [doc.lower().split() for doc in documents]
        
        # Compute average document length
        self.avgdl = sum(len(doc) for doc in self.tokenised_docs) / self.N
        
        # Compute document frequency for each word
        self.df = {}
        for doc in self.tokenised_docs:
            for word in set(doc):
                self.df[word] = self.df.get(word, 0) + 1
    
    def idf(self, word):
        """
        BM25 IDF formula:
        log( (N - df(w) + 0.5) / (df(w) + 0.5) + 1 )
        """
        n = self.df.get(word, 0)
        return math.log((self.N - n + 0.5) / (n + 0.5) + 1)
    
    def score_doc(self, query, doc_idx):
        """
        Compute BM25 score between a query and a specific document.
        """
        doc   = self.tokenised_docs[doc_idx]
        freq  = Counter(doc)
        dl    = len(doc)
        total = 0
        
        for word in query.lower().split():
            f   = freq.get(word, 0)          # raw count in doc
            idf = self.idf(word)             # global rarity
            
            # BM25 term frequency component (with saturation + length norm)
            tf_bm25 = (f * (self.k + 1)) / (
                f + self.k * (1 - self.b + self.b * (dl / self.avgdl))
            )
            total += idf * tf_bm25
        
        return total
    
    def rank(self, query, top_k=None):
        """
        Rank all documents by BM25 score for a given query.
        Returns: list of (doc_idx, score) sorted by score descending
        """
        scores = [(i, self.score_doc(query, i)) for i in range(self.N)]
        scores.sort(key=lambda x: -x[1])
        return scores[:top_k] if top_k else scores


# Test it on the space agency example
documents = [
    "nasa launched the apollo mission to the moon",
    "isro achieved the cheapest successful moon mission chandrayaan",
    "esa operates many space projects across europe"
]

bm25 = BM25(documents)
query = "space agency with a successful yet cheapest moon mission"

results = bm25.rank(query)
print("BM25 Ranking:")
for rank, (idx, score) in enumerate(results, 1):
    print(f"  {rank}. (BM25={score:.4f}) {documents[idx]}")

print(f"\nCorpus stats:")
print(f"  Avg doc length: {bm25.avgdl:.1f} words")
print(f"  Vocabulary size: {len(bm25.df)} words")

---
## 🔬 Part 4: Using rank-bm25 Library (Production)

In [ ]:
# ── rank-bm25 library — production-style BM25 ──

knowledge_base = [
    "NASA launched Apollo 11 and sent the first humans to the Moon in 1969.",
    "ISRO's Chandrayaan proved the Moon has water ice and was the cheapest lunar mission.",
    "SpaceX developed reusable Falcon 9 rockets to dramatically reduce launch costs.",
    "The Hubble Space Telescope has been orbiting Earth and capturing deep space images.",
    "Mars rovers like Curiosity and Perseverance have explored Mars since 2012.",
    "ESA's Rosetta probe successfully landed a robot on a comet for the first time.",
    "The James Webb Space Telescope can observe galaxies formed just after the Big Bang.",
]

# Tokenise (rank-bm25 expects pre-tokenised documents)
tokenised_kb = [doc.lower().split() for doc in knowledge_base]

# Create BM25 index
bm25_index = BM25Okapi(tokenised_kb)

def bm25_search(query_text, bm25_index, documents, top_k=3):
    """
    Search using BM25 Okapi ranking.
    
    Parameters:
        query_text : str  — search query
        bm25_index : fitted BM25Okapi model
        documents  : list[str] — original documents
        top_k      : int  — number of results
    """
    query_tokens = query_text.lower().split()
    scores = bm25_index.get_scores(query_tokens)
    top_idx = np.argsort(-scores)[:top_k]
    return [(scores[i], documents[i]) for i in top_idx]

# Test queries
test_queries = [
    "cheapest moon mission",
    "reusable rockets reduce cost",
    "human mission to Mars",
    "telescope observe galaxies",
]

for q in test_queries:
    print(f"\n🔍 BM25 Query: {q!r}")
    for i, (score, doc) in enumerate(bm25_search(q, bm25_index, knowledge_base), 1):
        print(f"  {i}. (score={score:.4f}) {doc}")

---
## 🔬 Part 5: TF-IDF vs BM25 — Direct Comparison

In [ ]:
# ── Direct comparison: where BM25 beats TF-IDF ──

# Edge case 1: repetition abuse
spam_corpus = [
    "buy cheap cheap cheap cheap cheap products cheap cheap cheap",  # Doc spams "cheap"
    "find the best deals and buy quality items at reasonable prices",  # More natural
    "cheap deals on quality items that you want to buy online",        # Balanced
]

spam_query = "cheap buy deals"

# TF-IDF
vec_spam = TfidfVectorizer()
mat_spam = vec_spam.fit_transform(spam_corpus)
qv_spam  = vec_spam.transform([spam_query])
tfidf_scores_spam = cosine_similarity(qv_spam, mat_spam).flatten()

# BM25
bm25_spam = BM25(spam_corpus)
bm25_scores_spam = [bm25_spam.score_doc(spam_query, i) for i in range(len(spam_corpus))]

print("Edge Case: Repetition Spam")
print(f"Query: {spam_query!r}\n")
print(f"{'Doc':<4} {'TF-IDF':>10} {'BM25':>10}  Text")
print('-' * 90)
for i, (ts, bs) in enumerate(zip(tfidf_scores_spam, bm25_scores_spam)):
    print(f"  {i+1}   {ts:>10.4f} {bs:>10.4f}  {spam_corpus[i][:60]}...")

tfidf_winner = np.argmax(tfidf_scores_spam) + 1
bm25_winner  = np.argmax(bm25_scores_spam) + 1
print(f"\n  TF-IDF winner: Doc {tfidf_winner} {'← spam wins! ❌' if tfidf_winner==1 else '✅'}")
print(f"  BM25 winner:   Doc {bm25_winner} {'← spam wins! ❌' if bm25_winner==1 else '✅'} (BM25 is more robust)'")

In [ ]:
# ── Visual comparison on the full knowledge base ──

comparison_query = "cheapest lunar mission ISRO moon"

# TF-IDF scores
vec = TfidfVectorizer(stop_words='english')
mat = vec.fit_transform(knowledge_base)
qv  = vec.transform([comparison_query])
tfidf_s = cosine_similarity(qv, mat).flatten()
# Normalise BM25 for comparison
bm25_kb  = BM25(knowledge_base)
bm25_raw = np.array([bm25_kb.score_doc(comparison_query, i) for i in range(len(knowledge_base))])
bm25_s   = (bm25_raw - bm25_raw.min()) / (bm25_raw.max() - bm25_raw.min() + 1e-9)

labels = [f"D{i+1}" for i in range(len(knowledge_base))]
x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - 0.2, tfidf_s, 0.4, label='TF-IDF (cosine)', color='#E74C3C', alpha=0.8)
ax.bar(x + 0.2, bm25_s,  0.4, label='BM25 (normalised)', color='#27AE60', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Retrieval Score')
ax.set_title(f'TF-IDF vs BM25 Scores\nQuery: {comparison_query!r}')
ax.legend()

# Annotate docs
for i, doc in enumerate(knowledge_base):
    ax.annotate(doc[:30]+'..',  xy=(i, -0.05), ha='center', va='top',
                fontsize=6, rotation=30, xycoords=('data', 'axes fraction'))

plt.tight_layout()
plt.savefig('tfidf_vs_bm25.png', dpi=120, bbox_inches='tight')
plt.show()
print("💾 Saved: tfidf_vs_bm25.png")

### ✏️ Exercise 5.1 — Tune k and b parameters

In [ ]:
# ✏️ Exercise 5.1 — How do k and b affect rankings?
# Experiment with different values and observe how rankings change.

# TODO: Try changing these values
k_value = 1.5   # Try: 0.5, 1.5, 3.0
b_value = 0.75  # Try: 0.0, 0.5, 1.0

bm25_tuned = BM25(knowledge_base, k=k_value, b=b_value)
q = "cheapest lunar mission ISRO moon"

print(f"BM25 with k={k_value}, b={b_value}")
print(f"Query: {q!r}\n")
for rank, (idx, score) in enumerate(bm25_tuned.rank(q), 1):
    print(f"  {rank}. (score={score:.4f}) {knowledge_base[idx]}")

print("\n💡 Try k=0.5 — does the spam-robustness improve?")
print("   Try b=0.0 — does a very long document win unfairly?")

---
## 📋 Summary: TF-IDF vs BM25

| Feature | TF-IDF | BM25 |
|---------|--------|------|
| Term repetition | Linear growth (spammable) | Saturates (robust) |
| Length normalisation | Simple division | Relative to corpus avgdl |
| Tunable parameters | None | k (saturation), b (length norm) |
| Default choice? | Legacy baseline | **Industry standard** |
| Used in production? | Some older systems | Elasticsearch, Solr, Lucene default |

### When to use BM25?
- **Always prefer BM25 over TF-IDF** for keyword search — it is strictly better
- BM25 is the **keyword component** in Hybrid RAG pipelines
- Well-suited for: web search, document retrieval, Q&A systems

In [ ]:
print("="*60)
print("EXERCISE 02 — BM25: COMPLETE REFERENCE")
print("="*60)
print()
print("FORMULA:")
print("  IDF(w)     = log( (N - df(w) + 0.5) / (df(w) + 0.5) + 1 )")
print("  TF_bm25    = f(w,d)×(k+1) / (f(w,d) + k×(1 - b + b×|d|/avgdl))")
print("  BM25(w,d)  = IDF(w) × TF_bm25")
print()
print("PARAMETERS:")
print("  k=1.5  → controls term saturation speed")
print("  b=0.75 → controls length normalisation strength")
print()
print("IMPROVEMENTS OVER TF-IDF:")
print("  1. Term saturation — diminishing returns for repetition")
print("  2. Length normalisation — longer docs penalised relative to avgdl")
print("="*60)